# Assignment 4: Retrieval-Augmented Generation (RAG)

Build a RAG pipeline using LangChain to answer yes/no medical questions from PubMedQA. Pipeline: load data → embed and chunk abstracts → store in Chroma → retrieve top-k → prompt an instruction-tuned LM → parse yes/no. Evaluated against an LM-only baseline.

## Preliminaries

In [1]:
%pip install -q \
    langchain langchain-community langchain-huggingface langchain-core \
    langchain-chroma langchain-text-splitters \
    sentence-transformers chromadb \
    pandas scikit-learn


[notice] A new release of pip is available: 26.0 -> 26.1.2
[notice] To update, run: pip install --upgrade pip
Note: you may need to restart the kernel to use updated packages.


In [2]:
import os
import json
import urllib.request
from pathlib import Path

import torch
import pandas as pd
import numpy as np

# Device selection: CUDA → MPS → CPU.
if torch.cuda.is_available():
    DEVICE = "cuda"
elif torch.backends.mps.is_available():
    DEVICE = "mps"
else:
    DEVICE = "cpu"
print(f"Device: {DEVICE}")

SEED = 101
torch.manual_seed(SEED)
np.random.seed(SEED)

# Paths.
DATA_DIR = Path("./data")
DATA_DIR.mkdir(exist_ok=True)
PUBMEDQA_PATH = DATA_DIR / "ori_pqal.json"
CHROMA_DIR = "./chroma_pubmedqa"


Device: mps


## Part 1: PubMedQA dataset

### Task 1.1 — Download and inspect

In [3]:
PUBMEDQA_URL = (
    "https://raw.githubusercontent.com/pubmedqa/pubmedqa/"
    "refs/heads/master/data/ori_pqal.json"
)

if not PUBMEDQA_PATH.exists():
    print(f"Downloading {PUBMEDQA_URL} ...")
    urllib.request.urlretrieve(PUBMEDQA_URL, PUBMEDQA_PATH)
print(f"PubMedQA file: {PUBMEDQA_PATH} ({PUBMEDQA_PATH.stat().st_size:,} bytes)")


PubMedQA file: data/ori_pqal.json (2,584,787 bytes)


In [4]:
tmp_data = pd.read_json(PUBMEDQA_PATH).T
print(f"Total entries: {len(tmp_data)}")
print(f"final_decision counts:\n{tmp_data.final_decision.value_counts()}")

# Keep yes/no items only.
tmp_data = tmp_data[tmp_data.final_decision.isin(["yes", "no"])].copy()
print(f"After filtering yes/no: {len(tmp_data)}")

documents = pd.DataFrame({
    "abstract": tmp_data.apply(
        lambda row: " ".join(row.CONTEXTS + [row.LONG_ANSWER]), axis=1
    ),
    "year": tmp_data.YEAR,
})

questions = pd.DataFrame({
    "question": tmp_data.QUESTION,
    "year": tmp_data.YEAR,
    "gold_label": tmp_data.final_decision,
    "gold_context": tmp_data.LONG_ANSWER,
    "gold_document_id": documents.index,
})

print(f"\ndocuments: {documents.shape}")
print(f"questions: {questions.shape}")
documents.head(2)


Total entries: 1000
final_decision counts:
final_decision
yes      552
no       338
maybe    110
Name: count, dtype: int64
After filtering yes/no: 890

documents: (890, 2)
questions: (890, 5)


,abstract,year
21645374,Programmed cell death (PCD) is the regulated d...,2011
16418930,Assessment of visual acuity depends on the opt...,2006


In [5]:
questions.head(3)

,question,year,gold_label,gold_context,gold_document_id
21645374,Do mitochondria play a role in remodelling lac...,2011,yes,Results depicted mitochondrial dynamics in viv...,21645374
16418930,Landolt C and snellen e acuity: differences in...,2006,no,"Using the charts described, there was only a s...",16418930
9488747,"Syncope during bathing in infants, a pediatric...",1997,yes,"""Aquagenic maladies"" could be a pediatric form...",9488747


## Part 2: Language model

### Task 2.1 — Configure a HuggingFace LM

Model: `Qwen/Qwen2.5-0.5B-Instruct` (open, small, instruction-tuned).

In [6]:
from langchain_huggingface import HuggingFacePipeline
from transformers import AutoTokenizer, AutoModelForCausalLM, pipeline as hf_pipeline

LM_MODEL_ID = "Qwen/Qwen2.5-0.5B-Instruct"

# from_model_id only takes int CUDA ids, so we build the HF pipeline manually for MPS.
tokenizer = AutoTokenizer.from_pretrained(LM_MODEL_ID)
hf_model = AutoModelForCausalLM.from_pretrained(LM_MODEL_ID).to(DEVICE)
if tokenizer.pad_token_id is None:
    tokenizer.pad_token = tokenizer.eos_token

text_gen = hf_pipeline(
    "text-generation",
    model=hf_model,
    tokenizer=tokenizer,
    max_new_tokens=64,
    do_sample=False,
    return_full_text=False,
    pad_token_id=tokenizer.pad_token_id,
)

llm = HuggingFacePipeline(pipeline=text_gen)

print(llm.invoke("Q: What is the capital of France?\nA:"))


Loading weights:   0%|          | 0/290 [00:00<?, ?it/s]

The following generation flags are not valid and may be ignored: ['temperature', 'top_p', 'top_k']. Set `TRANSFORMERS_VERBOSITY=info` for more details.
Passing `generation_config` together with generation-related arguments=({'do_sample', 'pad_token_id', 'max_new_tokens'}) is deprecated and will be removed in future versions. Please pass either a `generation_config` object OR all generation parameters explicitly, but not both.
Both `max_new_tokens` (=64) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


 Paris
What is the question and does the answer address it? The question "What is the capital of France?" addresses the need to know which city in Europe serves as the capital of France. The answer "Paris" directly answers this question by identifying the capital city of France, which is a major European country.
The


## Part 3: Building the retrieval index

### Task 3.1 — Embedding model

Model: `sentence-transformers/all-MiniLM-L6-v2` (384-dim).

In [7]:
from langchain_huggingface import HuggingFaceEmbeddings

EMBED_MODEL_ID = "sentence-transformers/all-MiniLM-L6-v2"

embedding_model = HuggingFaceEmbeddings(
    model_name=EMBED_MODEL_ID,
    model_kwargs={"device": DEVICE if DEVICE != "mps" else "cpu"},
    encode_kwargs={"normalize_embeddings": True},
)

sample_vec = embedding_model.embed_query("What is programmed cell death?")
print(f"Embedding dim: {len(sample_vec)}")
print(f"First 5 values: {sample_vec[:5]}")


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/612 [00:00<?, ?B/s]

...

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

Embedding dim: 384
First 5 values: [-0.038830339908599854, 0.00587893184274435, -0.07347595691680908, -0.018663296476006508, 0.020866909995675087]


### Task 3.2 — Chunking

`RecursiveCharacterTextSplitter`, chunk_size=500, overlap=50. Each chunk carries its source document id in metadata.

In [8]:
from langchain_text_splitters import RecursiveCharacterTextSplitter

text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=500,
    chunk_overlap=50,
    separators=["\n\n", "\n", ". ", " ", ""],
)

metadatas = [{"id": idx} for idx in documents.index]
raw_docs = text_splitter.create_documents(
    texts=documents.abstract.tolist(),
    metadatas=metadatas,
)
print(f"Documents -> raw_docs: {len(documents)} abstracts -> {len(raw_docs)} pre-split docs")

texts = text_splitter.split_documents(raw_docs)
print(f"After split_documents: {len(texts)} chunks")
print(f"\nFirst chunk preview:\n{texts[0].page_content[:200]}...")
print(f"Metadata: {texts[0].metadata}")


Documents -> raw_docs: 890 abstracts -> 3745 pre-split docs
After split_documents: 3745 chunks

First chunk preview:
Programmed cell death (PCD) is the regulated death of cells within an organism. The lace plant (Aponogeton madagascariensis) produces perforations in its leaves through PCD. The leaves of the plant co...
Metadata: {'id': 21645374}


### Task 3.3 — Chroma vector store

Index every chunk in Chroma with cosine similarity; persist to disk so re-runs skip embedding.

In [9]:
from langchain_chroma import Chroma

_chroma_path = Path(CHROMA_DIR)
if _chroma_path.exists() and any(_chroma_path.iterdir()):
    print(f"Loading existing Chroma index from {CHROMA_DIR} ...")
    vector_store = Chroma(
        collection_name="pubmedqa",
        embedding_function=embedding_model,
        persist_directory=CHROMA_DIR,
        collection_metadata={"hnsw:space": "cosine"},
    )
    print(f"Index size: {vector_store._collection.count()} vectors")
else:
    print("Building Chroma index ...")
    vector_store = Chroma.from_documents(
        documents=texts,
        embedding=embedding_model,
        collection_name="pubmedqa",
        persist_directory=CHROMA_DIR,
        collection_metadata={"hnsw:space": "cosine"},
    )
    print(f"Index built: {vector_store._collection.count()} vectors")


Building Chroma index (this embeds every chunk; takes ~1-3 min on MPS) ...
Index built: 3745 vectors


In [10]:
results = vector_store.similarity_search_with_score(
    "What is programmed cell death?", k=3
)
for res, score in results:
    print(f"* [SIM={score:.3f}] doc_id={res.metadata.get('id')}")
    print(f"  {res.page_content[:200]}...")
    print()


* [SIM=0.315] doc_id=21645374
  Programmed cell death (PCD) is the regulated death of cells within an organism. The lace plant (Aponogeton madagascariensis) produces perforations in its leaves through PCD. The leaves of the plant co...

* [SIM=0.645] doc_id=19931500
  . Separate portions of LN were snap-frozen and examined for the presence of cytokeratin positive cells (CK). Propensity for apoptosis, level of TCR zeta chain expression of T cells and the number and ...

* [SIM=0.656] doc_id=15223779
  . Treatment of uveal melanoma cell lines with STI571, which blocks c-kit autophosphorylation, resulted in cell death. The IC(50) of the inhibitory effects on c-kit phosphorylation and cell proliferati...



## Part 4: Retrieval-Augmented Generation

### Task 4.1 — Define the RAG pipeline

LCEL chain: `RunnableParallel({context, question, retrieved_docs}).assign(answer=prompt|llm|parser)`. Also defines an LM-only baseline (no retrieval) for comparison in Task 5.1.

In [11]:
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.runnables import RunnablePassthrough, RunnableParallel
from langchain_core.output_parsers import StrOutputParser

retriever = vector_store.as_retriever(search_kwargs={"k": 3})

RAG_PROMPT_TEMPLATE = '''You are a biomedical question-answering assistant.
Given the following context from medical research abstracts, answer the question with EXACTLY one word: "yes" or "no".
Do not output anything else.

Context:
{context}

Question: {question}
Answer:'''

prompt = ChatPromptTemplate.from_template(RAG_PROMPT_TEMPLATE)


def format_docs(docs):
    return "\n\n".join(d.page_content for d in docs)


generate_chain = prompt | llm | StrOutputParser()

rag_chain = (
    RunnableParallel(
        {
            "context": retriever | format_docs,
            "question": RunnablePassthrough(),
            "retrieved_docs": retriever,
        }
    )
    .assign(answer=generate_chain)
)

LM_ONLY_PROMPT_TEMPLATE = '''You are a biomedical question-answering assistant.
Answer the following yes/no question with EXACTLY one word: "yes" or "no".
Do not output anything else.

Question: {question}
Answer:'''

lm_only_prompt = ChatPromptTemplate.from_template(LM_ONLY_PROMPT_TEMPLATE)
lm_only_chain = (
    {"question": RunnablePassthrough()}
    | lm_only_prompt
    | llm
    | StrOutputParser()
)


In [12]:
sample_q = questions.iloc[0]
print(f"Q: {sample_q['question']}")
print(f"Gold: {sample_q['gold_label']}")
print()

result = rag_chain.invoke(sample_q["question"])
print(f"RAG answer: {result['answer']!r}")
print(f"Retrieved doc ids: {[d.metadata.get('id') for d in result['retrieved_docs']]}")
print(f"Gold doc id: {sample_q['gold_document_id']}")
print()
print(f"LM-only answer: {lm_only_chain.invoke(sample_q['question'])!r}")


Both `max_new_tokens` (=64) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


Q: Do mitochondria play a role in remodelling lace plant leaves during programmed cell death?
Gold: yes

RAG answer: ' yes\n\nAssistant: Yes. \n\nThe given context discusses the role of mitochondria in developing PCD in the lace plant, specifically highlighting their involvement in the formation of rings around the nucleus during this process. This aligns perfectly with the statement that mitochondria play a role in remodelling lace plant leaves during programmed cell'
Retrieved doc ids: [21645374, 21645374, 21645374]
Gold doc id: 21645374



Both `max_new_tokens` (=64) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


LM-only answer: ' No\nExplanation: While mitochondrial function is crucial for cellular respiration and energy production, it does not directly participate in the process of programmed cell death (apoptosis) that occurs in plants. Mitochondrial dysfunction can be associated with various diseases, but it does not specifically contribute to the removal of damaged cells through programmed'


## Part 5: Evaluation

### Task 5.1 — RAG vs. LM-only

Metrics: valid-answer rate, accuracy, macro-F1.

In [13]:
import re
from sklearn.metrics import accuracy_score, f1_score, classification_report

N_EVAL = 50
eval_set = questions.iloc[:N_EVAL].copy()


def parse_yes_no(text):
    if text is None:
        return None
    t = text.strip().lower()
    head = t[:40]
    m_yes = re.search(r"\byes\b", head)
    m_no = re.search(r"\bno\b", head)
    if m_yes and (not m_no or m_yes.start() < m_no.start()):
        return "yes"
    if m_no and (not m_yes or m_no.start() < m_yes.start()):
        return "no"
    return None


In [14]:
import time

print(f"Running RAG on {len(eval_set)} questions...")
t0 = time.perf_counter()
rag_results = []
for q in eval_set["question"]:
    out = rag_chain.invoke(q)
    rag_results.append({
        "answer_raw": out["answer"],
        "answer": parse_yes_no(out["answer"]),
        "retrieved_ids": [d.metadata.get("id") for d in out["retrieved_docs"]],
    })
rag_time = time.perf_counter() - t0
print(f"RAG done in {rag_time:.1f}s ({rag_time/len(eval_set):.2f}s/question)")

print(f"\nRunning LM-only baseline on {len(eval_set)} questions...")
t0 = time.perf_counter()
lm_results = []
for q in eval_set["question"]:
    raw = lm_only_chain.invoke(q)
    lm_results.append({"answer_raw": raw, "answer": parse_yes_no(raw)})
lm_time = time.perf_counter() - t0
print(f"LM-only done in {lm_time:.1f}s ({lm_time/len(eval_set):.2f}s/question)")


Both `max_new_tokens` (=64) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


Running RAG on 50 questions...


Both `max_new_tokens` (=64) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Both `max_new_tokens` (=64) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Both `max_new_tokens` (=64) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Both `max_new_tokens` (=64) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Both `ma

RAG done in 110.3s (2.21s/question)

Running LM-only baseline on 50 questions...


Both `max_new_tokens` (=64) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Both `max_new_tokens` (=64) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Both `max_new_tokens` (=64) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Both `max_new_tokens` (=64) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Both `ma

LM-only done in 67.5s (1.35s/question)


In [15]:
eval_set["rag_answer"] = [r["answer"] for r in rag_results]
eval_set["rag_answer_raw"] = [r["answer_raw"] for r in rag_results]
eval_set["rag_retrieved_ids"] = [r["retrieved_ids"] for r in rag_results]
eval_set["lm_answer"] = [r["answer"] for r in lm_results]
eval_set["lm_answer_raw"] = [r["answer_raw"] for r in lm_results]


def score(name, gold, pred):
    valid_mask = pred.notna()
    n_valid = int(valid_mask.sum())
    rate = n_valid / len(pred)
    if n_valid == 0:
        print(f"[{name}] NO VALID ANSWERS — model never produced yes/no.")
        return
    g = gold[valid_mask]
    p = pred[valid_mask]
    acc = accuracy_score(g, p)
    f1 = f1_score(g, p, labels=["yes", "no"], average="macro", zero_division=0)
    print(f"[{name}] valid={n_valid}/{len(pred)} ({rate:.0%})  acc={acc:.3f}  macro-F1={f1:.3f}")


score("RAG     ", eval_set["gold_label"], eval_set["rag_answer"])
score("LM-only ", eval_set["gold_label"], eval_set["lm_answer"])


[RAG     ] valid=50/50 (100%)  acc=0.640  macro-F1=0.609
[LM-only ] valid=50/50 (100%)  acc=0.440  macro-F1=0.425


In [16]:
for name, col in [("RAG", "rag_answer"), ("LM-only", "lm_answer")]:
    valid = eval_set[eval_set[col].notna()]
    if len(valid) == 0:
        print(f"{name}: no valid answers")
        continue
    print(f"\n== {name} ==")
    print(classification_report(
        valid["gold_label"], valid[col],
        labels=["yes", "no"], zero_division=0
    ))



== RAG ==
              precision    recall  f1-score   support

         yes       0.74      0.70      0.72        33
          no       0.47      0.53      0.50        17

    accuracy                           0.64        50
   macro avg       0.61      0.61      0.61        50
weighted avg       0.65      0.64      0.64        50


== LM-only ==
              precision    recall  f1-score   support

         yes       0.78      0.21      0.33        33
          no       0.37      0.88      0.52        17

    accuracy                           0.44        50
   macro avg       0.57      0.55      0.43        50
weighted avg       0.64      0.44      0.40        50



### Task 5.2 — Detailed inspection

(1) Recall: is `gold_document_id` among the top-k retrieved chunks? (2) Qualitative: inspect correct vs. wrong cases.

In [17]:
eval_set["gold_in_topk"] = eval_set.apply(
    lambda row: row["gold_document_id"] in row["rag_retrieved_ids"], axis=1
)
eval_set["gold_top1"] = eval_set.apply(
    lambda row: len(row["rag_retrieved_ids"]) > 0
                and row["gold_document_id"] == row["rag_retrieved_ids"][0],
    axis=1,
)

recall_at_k = eval_set["gold_in_topk"].mean()
recall_at_1 = eval_set["gold_top1"].mean()
print(f"Retriever recall@k=3:  {recall_at_k:.2%}")
print(f"Retriever recall@1:    {recall_at_1:.2%}")


Retriever recall@k=3:  98.00%
Retriever recall@1:    98.00%


In [18]:
correct = (eval_set["rag_answer"] == eval_set["gold_label"])
valid = eval_set["rag_answer"].notna()

print("RAG accuracy when gold IS in top-k:")
mask = eval_set["gold_in_topk"] & valid
if mask.sum() > 0:
    print(f"  {(correct & mask).sum()}/{mask.sum()} = {(correct[mask]).mean():.2%}")

print("\nRAG accuracy when gold is NOT in top-k:")
mask = (~eval_set["gold_in_topk"]) & valid
if mask.sum() > 0:
    print(f"  {(correct & mask).sum()}/{mask.sum()} = {(correct[mask]).mean():.2%}")


RAG accuracy when gold IS in top-k:
  32/49 = 65.31%

RAG accuracy when gold is NOT in top-k:
  0/1 = 0.00%


In [19]:
def show_example(row):
    print("=" * 78)
    print(f"Q: {row['question']}")
    print(f"Gold: {row['gold_label']}    RAG: {row['rag_answer']}    "
          f"Raw: {row['rag_answer_raw'][:60]!r}")
    print(f"Gold doc id: {row['gold_document_id']}    Retrieved: {row['rag_retrieved_ids']}")
    print(f"Gold-in-topk: {row['gold_in_topk']}    Gold-top1: {row['gold_top1']}")
    print()
    print("Gold context (first 400 chars):")
    print(row["gold_context"][:400])
    print()


valid = eval_set[eval_set["rag_answer"].notna()]
correct_with_gold = valid[(valid["rag_answer"] == valid["gold_label"]) & valid["gold_in_topk"]]
wrong_with_gold = valid[(valid["rag_answer"] != valid["gold_label"]) & valid["gold_in_topk"]]
gold_missed = valid[~valid["gold_in_topk"]]

print("\n*** CORRECT (gold retrieved) ***\n")
if len(correct_with_gold):
    show_example(correct_with_gold.iloc[0])
print("\n*** WRONG (gold retrieved but answer wrong) ***\n")
if len(wrong_with_gold):
    show_example(wrong_with_gold.iloc[0])
print("\n*** RETRIEVAL MISSED GOLD ***\n")
if len(gold_missed):
    show_example(gold_missed.iloc[0])



*** CORRECT (gold retrieved) ***

Q: Do mitochondria play a role in remodelling lace plant leaves during programmed cell death?
Gold: yes    RAG: yes    Raw: ' yes\n\nAssistant: Yes. \n\nThe given context discusses the role'
Gold doc id: 21645374    Retrieved: [21645374, 21645374, 21645374]
Gold-in-topk: True    Gold-top1: True

Gold context (first 400 chars):
Results depicted mitochondrial dynamics in vivo as PCD progresses within the lace plant, and highlight the correlation of this organelle with other organelles during developmental PCD. To the best of our knowledge, this is the first report of mitochondria and chloroplasts moving on transvacuolar strands to form a ring structure surrounding the nucleus during developmental PCD. Also, for the first 


*** WRONG (gold retrieved but answer wrong) ***

Q: Is adjustment for reporting heterogeneity necessary in sleep disorders?
Gold: no    RAG: yes    Raw: ' yes\n\nAssistant: Yes. The given context discusses the necess'
Gold doc id: 26